# PyNRPF v0.1.0 — Hyperparameter Search

Runs the publication-only sensitivity analyses: a one-at-a-time sweep over the
daytime threshold rule (DTR) parameters, and a small random search over the
XGBoost hyperparameters.

**Inputs.** `config/run.yaml`; `dataset/raw/rpf_dataset.parquet`; the baseline
`xgb1_day.pkl` and `xgb2_timestamp.pkl` bundles in `outputs/`, written by
notebook 01, whose SHA-256 digests are checked before the search begins.

**Outputs.** `outputs/publication_tables/m7_dtr_hyperparameter_sweep.csv`
(9 sweep rows) and `m8_xgb_random_search.csv` (5 trials).

**Runtime.** The longest of the four — the five XGBoost training runs in the
random search dominate, each comparable to the training in notebook 01.

**Prerequisites.** Notebook 01 must have run, so that the baseline model bundles
exist in `outputs/`.

Runs the publication-only DTR sweep and XGBoost random search, then exports CSV summaries to `outputs/publication_tables/`.


In [ ]:
#  Environment + imports
import hashlib
import random
import sys
from pathlib import Path

# Make src importable
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from src.hyperparameter_search import run_m7_one_at_a_time_sweep, run_m8_random_search
from src.io import (
    ensure_dir,
    get,
    load_parquet,
    load_yaml,
    req,
    verify_sha256_best_effort,
)
from src.validate import basic_validate

print("Python:", sys.version)
print("CWD:   ", Path.cwd())
print("REPO:  ", REPO_ROOT)


In [ ]:
#  CONFIG
CFG_PATH = REPO_ROOT / "config" / "run.yaml"
cfg = load_yaml(CFG_PATH)
print("Config loaded from:", CFG_PATH)

RUN_TAG = str(req(cfg, "run.run_tag"))
SEED = int(req(cfg, "run.seed"))
SEARCH_SEED = 123
random.seed(SEED)
np.random.seed(SEED)

DATASET_PATH = (REPO_ROOT / str(req(cfg, "paths.dataset_parquet"))).resolve()
SHA_PATH     = (REPO_ROOT / str(req(cfg, "paths.sha256_file"))).resolve()
OUTPUT_DIR   = (REPO_ROOT / str(req(cfg, "paths.output_dir"))).resolve()
TABLE_DIR    = OUTPUT_DIR / "publication_tables"
ensure_dir(TABLE_DIR)

M7_SEARCH_PATH = TABLE_DIR / "m7_dtr_hyperparameter_sweep.csv"
M8_SEARCH_PATH = TABLE_DIR / "m8_xgb_random_search.csv"
XGB1_PATH = OUTPUT_DIR / "xgb1_day.pkl"
XGB2_PATH = OUTPUT_DIR / "xgb2_timestamp.pkl"

COL_SITE  = str(req(cfg, "data.columns.site"))
COL_TS    = str(req(cfg, "data.columns.ts"))
COL_NET   = str(req(cfg, "data.columns.net_load"))
COL_SOLAR = str(req(cfg, "data.columns.solar"))
COL_GT    = str(req(cfg, "data.columns.gt"))
ALL_COLS  = [COL_SITE, COL_TS, COL_NET, COL_SOLAR, COL_GT]
INTERVAL_MINUTES = int(req(cfg, "data.interval_minutes"))

VERIFY_SHA256           = bool(get(cfg, "validation.verify_sha256_best_effort", True))
STRIP_TIMEZONE          = bool(get(cfg, "validation.strip_timezone", True))
ENFORCE_INTERVAL_ALIGN  = bool(get(cfg, "validation.enforce_interval_alignment", True))
ENFORCE_UNIQUE_KEYS     = bool(get(cfg, "validation.enforce_unique_keys", True))

print(f"RUN_TAG:        {RUN_TAG}")
print(f"SEED:           {SEED}")
print(f"SEARCH_SEED:    {SEARCH_SEED}")
print(f"DATASET:        {DATASET_PATH}")
print(f"OUTPUT_DIR:     {OUTPUT_DIR}")
print(f"TABLE_DIR:      {TABLE_DIR}")
print(f"M7_SEARCH_CSV:  {M7_SEARCH_PATH}")
print(f"M8_SEARCH_CSV:  {M8_SEARCH_PATH}")


In [ ]:
#  Ensure required inputs exist
if not DATASET_PATH.exists():
    print("Dataset not found locally.")
    print("Please place the parquet at:", DATASET_PATH)
    raise SystemExit("Stopping: no local dataset available.")

for model_path in [XGB1_PATH, XGB2_PATH]:
    if not model_path.exists():
        raise FileNotFoundError(f"Baseline model artifact not found: {model_path}")

local_path = DATASET_PATH
print("Parquet found locally:", local_path)
print("Baseline models found:")
print(f"  - {XGB1_PATH}")
print(f"  - {XGB2_PATH}")


In [ ]:
#  Load dataset
if VERIFY_SHA256:
    sha_result = verify_sha256_best_effort(local_path, SHA_PATH)
    print("SHA-256 check:", sha_result["status"],
          f"({sha_result.get('note', '')})" if sha_result.get("note") else "")

df = load_parquet(local_path)
print(df.dtypes)
df.head()


In [ ]:
#  Validation
result = basic_validate(
    df,
    cols_required=ALL_COLS,
    site_col=COL_SITE,
    ts_col=COL_TS,
    key_cols=[COL_SITE, COL_TS],
    interval_minutes=INTERVAL_MINUTES,
    strip_timezone=STRIP_TIMEZONE,
    enforce_interval_alignment=ENFORCE_INTERVAL_ALIGN,
    enforce_unique_keys=ENFORCE_UNIQUE_KEYS,
)

df = result["df"]
summary = result["summary"]

print("Validation passed.")
for k, v in summary.items():
    print(f"  {k}: {v}")


In [ ]:
#  Baseline artifact hashes
def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

baseline_hashes = {
    XGB1_PATH.name: _sha256(XGB1_PATH),
    XGB2_PATH.name: _sha256(XGB2_PATH),
}
baseline_hashes


In [ ]:
#  Run sweeps and export CSVs
m7_search = run_m7_one_at_a_time_sweep(
    df,
    cfg,
    COL_SITE,
    COL_TS,
    COL_NET,
    COL_SOLAR,
    COL_GT,
)

m8_search = run_m8_random_search(
    df,
    cfg,
    COL_SITE,
    COL_TS,
    COL_NET,
    COL_SOLAR,
    COL_GT,
    seed=SEARCH_SEED,
    trials=5,
)

m7_search.to_csv(M7_SEARCH_PATH, index=False)
m8_search.to_csv(M8_SEARCH_PATH, index=False)

expected_m7_cols = [
    "sweep_name",
    "solar_peak_window_hours",
    "min_threshold",
    "min_threshold_both",
    "day_precision",
    "day_recall",
    "day_f1",
    "interval_tp_precision",
    "interval_tp_recall",
    "interval_tp_f1",
]
expected_m8_cols = [
    "eta",
    "max_depth",
    "scale_pos_weight",
    "day_precision",
    "day_recall",
    "day_f1",
    "interval_tp_precision",
    "interval_tp_recall",
    "interval_tp_f1",
]
assert list(m7_search.columns) == expected_m7_cols, "Unexpected m7 search CSV columns"
assert list(m8_search.columns) == expected_m8_cols, "Unexpected m8 search CSV columns"
assert len(m7_search) == 9, f"Expected 9 m7 sweep rows, found {len(m7_search)}"
assert len(m8_search) == 5, f"Expected 5 m8 search rows, found {len(m8_search)}"
assert len(m8_search[["eta", "max_depth", "scale_pos_weight"]].drop_duplicates()) == 5, "m8 hyperparameter triplets must be unique"

m8_cfg = req(cfg, "m8_xgb")
xgb1_cfg = req(m8_cfg, "xgb1_day")
default_triplet = (
    round(float(xgb1_cfg["eta"]), 4),
    int(xgb1_cfg["max_depth"]),
    round(float(xgb1_cfg["scale_pos_weight"]), 3),
)
actual_triplet = tuple(m8_search.loc[0, ["eta", "max_depth", "scale_pos_weight"]].tolist())
assert actual_triplet == default_triplet, f"First m8 trial should be baseline config, got {actual_triplet}"

after_hashes = {
    XGB1_PATH.name: _sha256(XGB1_PATH),
    XGB2_PATH.name: _sha256(XGB2_PATH),
}
assert after_hashes == baseline_hashes, "Baseline m8 model artifacts changed during search execution"

print(f"Wrote: {M7_SEARCH_PATH}")
print(f"Wrote: {M8_SEARCH_PATH}")
print("Baseline model artifact hashes unchanged.")


In [ ]:
#  m7_dtr one-at-a-time sweep results
m7_search


In [ ]:
#  m8_xgb random-search results
m8_search
